# Audit DB Advanced: How to Detect Trouble Automatically

**Difficulty: Advanced | ~45 min | Requires Lab 5 (Metrics and Dashboards) done in this Supabase project**

Lab 5 turned a raw log into *metrics*: error rates, cost and latency, throughput. This lab takes the
last step up -- from "measure the log" to "let the log tell you when something is wrong." Every alert
is the same shape:

> An **alert** is a query that returns rows only when something is wrong. A healthy system returns an
> empty result; a problem returns the offending rows.

The **threshold** is the `WHERE`/`HAVING` clause, the **metric** is the input, and **"firing"** just
means "the result set is not empty." We will build that rule up from a fixed number to a *relative*
one -- an agent compared to its own past -- which is the conceptual peak of the lab.

---


In [1]:
!pip install python-dotenv==1.2.3 psycopg2-binary==2.9.12



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Run this first cell to install exactly the versions this lab was built and tested with.

- `python-dotenv` reads our secret `DATABASE_URL` out of the project's `.env` file.
- `psycopg2-binary` is the Postgres driver every audit lab has used so far.

Nothing else is needed -- all the detection (aggregation, bucketing, percentiles, comparisons)
happens inside Postgres, and we only *drive* it from Python.


### Step 1 &ndash; Connect to Postgres


In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv
import psycopg2

env_path = next((p / ".env" for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").exists()), None)
load_dotenv(env_path)
database_url = os.getenv("DATABASE_URL")
assert database_url, "DATABASE_URL not found - complete README Section 7 setup first."

connection = psycopg2.connect(database_url, connect_timeout=10)
cursor = connection.cursor()
cursor.execute("SELECT version();")
print(cursor.fetchone()[0])


PostgreSQL 17.6 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit


Identical to every lab before it: walk *up* from the current folder until `.env` is found, load
`DATABASE_URL`, open one connection, and confirm the server answers by printing its version.

Keep `cursor` and `connection` around -- we will reuse them for the whole notebook.


### Step 2 &ndash; Confirm the schema is here


In [3]:
# Fail-fast: confirm the Lab 1 + Lab 2 tables exist before we write against them.
cursor.execute("SELECT to_regclass('public.run')")
assert cursor.fetchone()[0] is not None, "run table missing - complete Lab 1 first."
cursor.execute("SELECT to_regclass('public.event')")
assert cursor.fetchone()[0] is not None, "event table missing - complete Lab 1 first."
for table in ["span", "tool_call", "guardrail_event"]:
    cursor.execute("SELECT to_regclass(%s)", (f"public.{table}",))
    assert cursor.fetchone()[0] is not None, f"{table} table missing - complete Lab 2 first."
print("Schema found: run, event, span, tool_call, guardrail_event.")


Schema found: run, event, span, tool_call, guardrail_event.


`to_regclass('public.run')` returns the table's identifier if it exists and `None` if it does not.
Any `None` means an earlier lab has not been run, and the assertions stop us before we start building
alerts against tables that are not there.

Our alerts read `run` (error rates, cost, latency), `span` (to reach child tables), and `tool_call`
(for the retry-storm model in Step 6), so the Lab 1 and Lab 2 tables must all be present.


### Step 3 &ndash; Seed a spread with planted anomalies


In [4]:
import secrets

# One tag for this entire run-through, so cleanup (Step 9) can find exactly these rows.
lab6_tag = secrets.token_hex(4)

# (logical_agent, status, total_cost, started_a_go, duration)
#   agent_name = pytest-lab6-<agent>-<tag>-<i>  -- split_part('-', 3) recovers the agent
#   Each agent plants ONE kind of trouble the alerts must catch; "calm" stays healthy.
seed_runs = [
    # calm: a healthy baseline -- nothing below should ever fire on these
    ("calm",    "success", 0.0010, "6 hours",   "5 seconds"),
    ("calm",    "success", 0.0010, "10 hours",  "6 seconds"),
    ("calm",    "success", 0.0010, "12 hours",  "7 seconds"),
    # noisy: an error-rate SPIKE (3 of 4 errored = 75%), all fast runs
    ("noisy",   "error",   0.0050, "5 hours",   "8 seconds"),
    ("noisy",   "success", 0.0030, "7 hours",   "9 seconds"),
    ("noisy",   "error",   0.0070, "9 hours",   "9 seconds"),
    ("noisy",   "error",   0.0040, "11 hours",  "9 seconds"),
    # pricey: one run wildly over the cost cap
    ("pricey",  "success", 0.0100, "14 hours",  "5 seconds"),
    ("pricey",  "success", 999.0,  "16 hours",  "6 seconds"),
    ("pricey",  "success", 0.0150, "18 hours",  "7 seconds"),
    # bursty: five runs landing in ONE tight window (volume spike, isolated bucket)
    ("bursty",  "success", 0.0010, "2 hours",   "5 seconds"),
    ("bursty",  "error",   0.0010, "2 hours",   "6 seconds"),
    ("bursty",  "success", 0.0010, "2 hours",   "7 seconds"),
    ("bursty",  "error",   0.0010, "2 hours",   "8 seconds"),
    ("bursty",  "success", 0.0010, "2 hours",   "9 seconds"),
    # laggy: one run with a clear latency outlier
    ("laggy",   "success", 0.0050, "20 hours",  "8 seconds"),
    ("laggy",   "success", 0.0050, "23 hours",  "9 seconds"),
    ("laggy",   "success", 0.0020, "26 hours",  "300 seconds"),
    # tricky: LOW baseline that then jumps -- a fixed threshold misses it,
    #   but a baseline comparison catches it (the Step 8 peak)
    ("tricky",  "success", 0.0020, "8 hours",   "5 seconds"),
    ("tricky",  "success", 0.0020, "10 hours",  "6 seconds"),
    ("tricky",  "success", 0.0020, "10 minutes", "4 seconds"),
    ("tricky",  "error",   0.0020, "15 minutes", "5 seconds"),
    ("tricky",  "success", 0.0020, "20 minutes", "6 seconds"),
    # loopy: a retry storm - one run repeats the same tool call, otherwise fast
    ("loopy",   "success", 0.0200, "6 hours",   "9 seconds"),
]

inserted = []
for i, (agent, status, cost, ago, duration) in enumerate(seed_runs):
    name = f"pytest-lab6-{agent}-{lab6_tag}-{i}"
    cursor.execute(
        "INSERT INTO run (agent_name, status, total_cost, started_at, ended_at) "
        "VALUES (%s, %s, %s, now() - (interval %s + interval %s), now() - interval %s) "
        "RETURNING run_id",
        (name, status, cost, ago, duration, ago),
    )
    run_id = cursor.fetchone()[0]
    cursor.execute(
        "INSERT INTO span (run_id, span_name, started_at, ended_at) "
        "VALUES (%s, %s, now() - (interval %s + interval %s), now() - interval %s) "
        "RETURNING span_id",
        (run_id, "handle_request", ago, duration, ago),
    )
    span_id = cursor.fetchone()[0]
    if agent == "loopy":
        # A retry storm: the same operation, attempted six times inside one span.
        for attempt in range(6):
            cursor.execute(
                "INSERT INTO tool_call (span_id, tool_name, success) VALUES (%s, %s, %s)",
                (span_id, "check_status", attempt % 3 != 0),
            )
    inserted.append(run_id)

connection.commit()
print(f"Seeded {len(inserted)} tagged runs under marker pytest-lab6-*-{lab6_tag}-* "
      f"(run ids {inserted[0]}..{inserted[-1]}).")


Seeded 24 tagged runs under marker pytest-lab6-*-5bed267f-* (run ids 1258..1281).


An alert is only meaningful if we control *what the data looks like when it is wrong*. So this seed
isn't random -- it plants each problem the alerts must catch, and keeps one agent (`calm`) fully
healthy as the contrast case:

- **noisy** spikes (3 of 4 runs errored) for the static error-rate threshold.
- **pricey** has one run costing `999.0` for the cost cap.
- **bursty** lands five runs inside one tight window for the volume threshold.
- **laggy** has one `300`-second run for the latency outliers.
- **tricky** has a low baseline that *then* jumps -- the case a fixed number cannot see, but a
  relative comparison can (Step 8).
- **loopy** repeats the same tool call six times for the retry storm (Step 6).
- **calm** stays quiet, so we can prove each alert returns **zero rows** on a healthy slice.

Every run carries the same private tag in `agent_name` (`pytest-lab6-<agent>-<tag>-<n>`), our marker
alone. `split_part('-', 3)` recovers the logical agent, exactly as Lab 5 did. The run's `started_at`
is pushed into the past by `started_a_go` and its latency equals `duration`, so `ended_at - started_at`
is exactly what we designed -- every threshold below is hand-checkable.


### Step 4 &ndash; The simplest alert: a static error-rate threshold


In [5]:
# The alert: agents whose error rate is above a fixed bar. Non-empty == fabricating.
error_threshold = 50.0
cursor.execute("""
    SELECT
        split_part(agent_name, '-', 3)                       AS agent_name,
        count(*)                                         AS total_runs,
        count(*) FILTER (WHERE status = 'error')         AS errored_runs,
        round(100.0 * count(*) FILTER (WHERE status = 'error') / count(*), 1) AS error_rate_pct
    FROM run
    WHERE agent_name LIKE 'pytest-lab6-%%'
    GROUP BY split_part(agent_name, '-', 3)
    HAVING 100.0 * count(*) FILTER (WHERE status = 'error') / count(*) > %s
    ORDER BY error_rate_pct DESC
""", (error_threshold,))
fired = cursor.fetchall()
print(f"Agents firing (error rate > {int(error_threshold)}%):")
if not fired:
    print("  (none - healthy)")
for agent, total, errored, pct in fired:
    print(f"  {agent}: {errored}/{total} runs errored = {pct}%")

# The empty case matters just as much: the SAME query over the healthy agent.
cursor.execute("""
    SELECT
        split_part(agent_name, '-', 3),
        count(*) FILTER (WHERE status = 'error')
    FROM run
    WHERE agent_name LIKE 'pytest-lab6-calm-%%'
    GROUP BY split_part(agent_name, '-', 3)
    HAVING 100.0 * count(*) FILTER (WHERE status = 'error') / count(*) > %s
""", (error_threshold,))
print("Same query, healthy slice: returned", cursor.rowcount, "rows -- calm is fine.")


Agents firing (error rate > 50%):
  noisy: 3/4 runs errored = 75.0%
Same query, healthy slice: returned 0 rows -- calm is fine.


Read Step 4 as the whole lab in miniature. The `HAVING` clause is the **threshold**; the grouped
error-rate query is the **metric**; and **firing** is simply "did any rows come back?"

- The first `cursor.execute` returns one row per agent whose error rate clears `50%`. Only **noisy**
  (75%) does -- `calm`, `pricey`, `laggy`, `tricky`, `bursty`, and `loopy` are all below it, so they
  do not appear. A non-empty result *is* the alert: it carries exactly the offending rows.
- The second half proves the healthy case. The **identical** query, narrowed to `pytest-lab6-calm-%`,
  returns `0` rows. When nothing is wrong, the alert is silent -- an empty result set is a *good*
  signal, not an error.

That symmetry -- fires on the anomaly, stays quiet on health -- is the template every later step
repeats, just with a smarter threshold.


### Step 5 &ndash; Cost and volume thresholds


In [6]:
# 5a) Runs over a cost cap -- a plain WHERE-style alert (row-level threshold).
cost_cap = 1.0
cursor.execute("""
    SELECT agent_name, total_cost, status
    FROM run
    WHERE agent_name LIKE 'pytest-lab6-%%'
      AND total_cost > %s
    ORDER BY total_cost DESC
""", (cost_cap,))
over_cost = cursor.fetchall()
print(f"Runs over the ${int(cost_cap)} cost cap: {len(over_cost)}")
for name, cost, status in over_cost:
    print(f"  {name}: ${cost} ({status})")

# 5b) Volume: more than a fixed number of runs inside one hour bucket.
max_per_hour = 4
cursor.execute("""
    SELECT date_trunc('hour', started_at) AS hour_bucket, count(*) AS runs
    FROM run
    WHERE agent_name LIKE 'pytest-lab6-%%'
    GROUP BY hour_bucket
    HAVING count(*) > %s
    ORDER BY runs DESC
""", (max_per_hour,))
bursty_hours = cursor.fetchall()
print(f"Hour buckets with more than {max_per_hour} runs: {len(bursty_hours)}")
for hour, runs in bursty_hours:
    print(f"  {hour.isoformat()}: {runs} runs")


Runs over the $1 cost cap: 1
  pytest-lab6-pricey-5bed267f-8: $999.000000 (success)
Hour buckets with more than 4 runs: 1
  2026-08-28T15:00:00+00:00: 5 runs


Two thresholds, one idea.

- **5a** is a *row-level* check -- `WHERE total_cost > 1.0`. Only **pricey**'s `999.0` run clears it.
  This is the cheapest kind of alert: it examines each run by itself, no grouping at all. It is exactly
  the "cost per run over a cap" a budget guardrails would flag.
- **5b** is a *grouped* check -- `date_trunc('hour', started_at)` snaps each run to its hour, then
  `HAVING count(*) > 4` fires when any one hour contains more than four runs. Only **bursty**'s five
  runs, all landing in a single tight window, trip it.

This is the first alert that needs the aggregate vocabulary from Lab 5: the *bucket* is the metric,
the `HAVING` clause is the threshold. Same shape as Step 4, but the thing being measured is a count
over time instead of a single field.


### Step 6 &ndash; Retry storms


In [7]:
# A retry storm = the same operation attempted many times. Group tool calls by
# (run, tool_name) and fire when one run repeats an operation N+ times.
max_attempts = 5
cursor.execute("""
    SELECT
        r.agent_name,
        tc.tool_name,
        count(*) AS attempts,
        count(*) FILTER (WHERE tc.success = false) AS failed_attempts
    FROM tool_call tc
    JOIN span s     ON s.span_id = tc.span_id
    JOIN run r      ON r.run_id  = s.run_id
    WHERE r.agent_name LIKE 'pytest-lab6-%%'
    GROUP BY r.agent_name, tc.tool_name
    HAVING count(*) >= %s
    ORDER BY attempts DESC
""", (max_attempts,))
retries = cursor.fetchall()
print(f"Retry storms (same operation >= {max_attempts} times in one run): {len(retries)}")
for name, tool, attempts, failed in retries:
    print(f"  {name}: '{tool}' attempted {attempts} times, {failed} failed")


Retry storms (same operation >= 5 times in one run): 1
  pytest-lab6-loopy-5bed267f-23: 'check_status' attempted 6 times, 2 failed


A **retry storm** is when an agent gets stuck: it keeps calling the same tool (say `check_status`)
over and over instead of giving up. That is worth alerting on for two reasons -- every retry burns
tokens and CPU (wasted cost), and it usually means a loop the agent cannot escape.

We reach `tool_call` through `span` (Lab 2), as Step 2 promised. Grouping by `r.agent_name,
tc.tool_name` folds all the calls of one operation per run together, and `HAVING count(*) >= 5`
fires when a single run attempted the same thing at least five times. Only **loopy**'s six
`check_status` calls trip it -- every other run has zero or one tool call, so nothing else appears.

The `FILTER (WHERE tc.success = false)` adds a useful extra column: *how many* of those attempts
failed. This is detection with a diagnosis attached, not just a flag.


### Step 7 &ndash; Latency outliers: fixed, then percentile-based


In [8]:
# 7a) Fixed latency budget -- a run that took longer than a hard limit.
latency_budget_s = 120
cursor.execute("""
    SELECT agent_name,
           round(EXTRACT(EPOCH FROM (ended_at - started_at)), 1) AS latency_s
    FROM run
    WHERE agent_name LIKE 'pytest-lab6-%%'
      AND EXTRACT(EPOCH FROM (ended_at - started_at)) > %s
    ORDER BY latency_s DESC
""", (latency_budget_s,))
over_budget = cursor.fetchall()
print(f"Runs over the {latency_budget_s}s fixed budget: {len(over_budget)}")
for name, latency_s in over_budget:
    print(f"  {name}: {latency_s}s")

# 7b) Percentile-based budget: the bar derives from THIS workload, not a guess.
cursor.execute("""
    WITH latencies AS (
        SELECT agent_name,
               EXTRACT(EPOCH FROM (ended_at - started_at)) AS latency_s
        FROM run
        WHERE agent_name LIKE 'pytest-lab6-%'
    ),
    p95 AS (
        SELECT percentile_cont(0.95) WITHIN GROUP (ORDER BY latency_s) AS p95_s
        FROM latencies
    )
    SELECT l.agent_name, round(l.latency_s::numeric, 1) AS latency_s, round(p.p95_s::numeric, 1) AS p95_s
    FROM latencies l, p95 p
    WHERE l.latency_s > p.p95_s
    ORDER BY l.latency_s DESC
""")
percentile_outliers = cursor.fetchall()
print("Runs above the workload's own 95th-percentile latency:")
if not percentile_outliers:
    print("  (none)")
for name, latency_s, p95_s in percentile_outliers:
    print(f"  {name}: {latency_s}s vs p95={p95_s}s")


Runs over the 120s fixed budget: 1
  pytest-lab6-laggy-5bed267f-17: 300.0s


Runs above the workload's own 95th-percentile latency:
  pytest-lab6-laggy-5bed267f-17: 300.0s vs p95=9.0s


Latency is a *distribution*, so there are two reasonable budgets -- and they answer different
questions.

- **7a (fixed)** sets a hard limit, `120s`. Only **laggy**'s `300s` run trips it. Simple to reason
  about, but *you* must pick the number, and a fixed guess goes out of date as the workload changes:
  a limit that fits this lab's fast runs would drown a team whose typical run takes minutes.
- **7b (percentile-based)** computes the bar from the data itself. A `CTE` first pulls every tagged
  run's latency, a second `CTE` derives `percentile_cont(0.95)`, and the outer `SELECT` keeps only
  runs *above* that self-derived threshold. Now the alert adapts: it flags whatever is unusually slow
  *for this workload* -- no hand-tuning, and it still catches laggy's `300s` outlier.

The lesson is the same one from Lab 5's p95: fixed thresholds need a human to maintain them;
percentile thresholds let the data tell you what "abnormal" means.


### Step 8 &ndash; Baseline deviation: compare each agent to its own past


In [9]:
# The peak: a RELATIVE alert. Each agent's recent error rate is compared to its
# own trailing baseline; it fires only when it deviates by more than a margin.
window = "60 minutes"
margin = 25.0
min_recent_runs = 2
cursor.execute("""
    WITH recent AS (
        SELECT split_part(agent_name, '-', 3) AS agent_name,
               count(*) AS recent_runs,
               count(*) FILTER (WHERE status = 'error') AS recent_errors
        FROM run
        WHERE agent_name LIKE 'pytest-lab6-%%'
          AND started_at >= now() - %s::interval
        GROUP BY split_part(agent_name, '-', 3)
    ),
    baseline AS (
        SELECT split_part(agent_name, '-', 3) AS agent_name,
               count(*) AS base_runs,
               count(*) FILTER (WHERE status = 'error') AS base_errors
        FROM run
        WHERE agent_name LIKE 'pytest-lab6-%%'
          AND started_at < now() - %s::interval
        GROUP BY split_part(agent_name, '-', 3)
    )
    SELECT
        r.agent_name,
        round(100.0 * r.recent_errors / r.recent_runs, 1) AS recent_rate_pct,
        round(100.0 * COALESCE(b.base_errors, 0) / NULLIF(b.base_runs, 0), 1) AS baseline_rate_pct,
        round(100.0 * r.recent_errors / r.recent_runs
              - 100.0 * COALESCE(b.base_errors, 0) / NULLIF(b.base_runs, 0), 1) AS deviation_pct
    FROM recent r
    LEFT JOIN baseline b ON b.agent_name = r.agent_name
    WHERE r.recent_runs >= %s
      AND (100.0 * r.recent_errors / r.recent_runs
           - 100.0 * COALESCE(b.base_errors, 0) / NULLIF(b.base_runs, 0)) > %s
    ORDER BY deviation_pct DESC
""", (window, window, min_recent_runs, margin))
fired = cursor.fetchall()
print(f"Agents deviating from their own baseline by > {int(margin)} points "
      f"(last {window}, >= {min_recent_runs} recent runs): {len(fired)}")
for agent, recent_rate, base_rate, dev in fired:
    print(f"  {agent}: recent {recent_rate}% vs baseline {base_rate}% (deviation +{dev})")

# 8c) Package an alert as a named, reusable check (a VIEW): an "alert-as-a-query".
cursor.execute("DROP VIEW IF EXISTS v_alerts_high_error_rate")
cursor.execute("""
    CREATE VIEW v_alerts_high_error_rate AS
    SELECT
        split_part(agent_name, '-', 3) AS agent_name,
        round(100.0 * count(*) FILTER (WHERE status = 'error') / count(*), 1) AS error_rate_pct
    FROM run
    WHERE agent_name LIKE 'pytest-lab6-%%'
    GROUP BY split_part(agent_name, '-', 3)
    HAVING 100.0 * count(*) FILTER (WHERE status = 'error') / count(*) > %s
""", (error_threshold,))
connection.commit()
cursor.execute("SELECT agent_name, error_rate_pct FROM v_alerts_high_error_rate")
print(f"8c) v_alerts_high_error_rate (a named alert-as-a-view) returns:")
for agent, pct in cursor.fetchall():
    print(f"    {agent}: {pct}%")


Agents deviating from their own baseline by > 25 points (last 60 minutes, >= 2 recent runs): 1
  tricky: recent 33.3% vs baseline 0.0% (deviation +33.3)


8c) v_alerts_high_error_rate (a named alert-as-a-view) returns:
    noisy: 75.0%


Every threshold so far was a *fixed number*: `> 50%`, `$ > 1`, `count > 4`, `>= 5`, `> 120s`. Step 8
throws all of those away and asks a harder question: *is this agent worse than it usually is for
ITSELF?*

Two `CTE`s split each agent's runs by time. **`recent`** holds runs from the last `60 minutes`; the
**trailing baseline** holds everything older. The outer `SELECT` computes each agent's recent error
rate versus its own baseline rate, and the `WHERE` fires only two conditions at once -- enough recent
runs to trust the sample (`>= 2`) **and** a deviation past the margin (`> 25` points). Only
**tricky** clears the bar: its recent runs sit within the window, its baseline is `0%`, and its
recent `33%` is a `33`-point leap. (Every other agent is skipped for a different reason -- `noisy`'s
runs are all *older* than the window, so it has no recent sample to compare.)

Now the payoff of seeding `tricky` the way we did: its recent rate is `33%`. That is *below* Step 4's
fixed `50%` bar, so the static alert stayed silent on it. But `33%` is a sharp jump from `tricky`'s own
`0%` baseline -- a `33`-point relative deviation. A **relative** alert catches "worse than normal for
THIS agent" even when no fixed threshold is crossed. That is the most powerful and the most subtle
kind of detection, because "normal" is defined per agent rather than globally.

**8c** closes the loop on the whole lab's through-line. Because an alert is just a query, we can name
it: `CREATE VIEW v_alerts_high_error_rate` packages Step 4's static check as a reusable, named alert.
Now "is an agent misbehaving?" is a one-line `SELECT * FROM v_alerts_high_error_rate` -- a check any
monitoring job could run on a schedule against the audit DB. It is dropped again in Step 9 so the
schema returns to its Lab 1 + Lab 2 shape.


### Step 9 &ndash; Cleanup and recap


In [10]:
# Drop the one view this lab created.
cursor.execute("DROP VIEW IF EXISTS v_alerts_high_error_rate")
connection.commit()

# Delete only this lab's tagged rows - children first (FK order), then parents.
cursor.execute(
    "DELETE FROM tool_call WHERE span_id IN "
    "(SELECT span_id FROM span WHERE run_id IN (SELECT run_id FROM run WHERE agent_name LIKE %s))",
    ("pytest-lab6-%",),
)
cursor.execute(
    "DELETE FROM guardrail_event WHERE span_id IN "
    "(SELECT span_id FROM span WHERE run_id IN (SELECT run_id FROM run WHERE agent_name LIKE %s))",
    ("pytest-lab6-%",),
)
cursor.execute(
    "DELETE FROM span WHERE run_id IN (SELECT run_id FROM run WHERE agent_name LIKE %s)",
    ("pytest-lab6-%",),
)
cursor.execute(
    "DELETE FROM event WHERE run_id IN (SELECT run_id FROM run WHERE agent_name LIKE %s)",
    ("pytest-lab6-%",),
)
cursor.execute("DELETE FROM run WHERE agent_name LIKE %s", ("pytest-lab6-%",))
connection.commit()

cursor.execute("SELECT count(*) FROM run WHERE agent_name LIKE %s", ("pytest-lab6-%",))
leftover = cursor.fetchone()[0]

cursor.close()
connection.close()

print(f"Cleanup done - {leftover} tagged runs left behind; view dropped.")
print("Lab 6 complete: alerts are queries that return rows only when something is wrong.")


Cleanup done - 0 tagged runs left behind; view dropped.
Lab 6 complete: alerts are queries that return rows only when something is wrong.


We leave the schema exactly as Lab 1 + Lab 2 shipped it:

- **Drop** the one view (`v_alerts_high_error_rate`) we created; `IF EXISTS` makes it safe to re-run.
- **Delete** only our `pytest-lab6-%` rows, children before parents so foreign keys are satisfied
  (`tool_call` and `guardrail_event` before `span`/`event`/`run`).
- **Verify** nothing tagged remains, then close the connection.

**What this lab proved:** every alert is the same idea -- a query whose empty/non-empty result is the
signal. We built that idea up from a fixed error-rate bar (Step 4), to row and volume thresholds
(Step 5), to a retry storm (Step 6), to fixed-then-percentile latency budgets (Step 7), and finally to
a *relative* baseline comparison that catches abnormal-for-this-agent when no fixed number does
(Step 8). Together Labs 1-6 form the built half of the module: **write -> model -> read -> filter ->
measure -> alert**. These are exactly the queries a real harness would run on a schedule against its
audit DB -- and Lab 7 is the half that protects that DB from tampering.
